# 33. Stage-wise Feature Separability 분석

encoder stage별 feature가 color, defect, stress/success label을 얼마나 잘 분리하는지 확인합니다.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch3_utils.py").exists():
    matches = list(Path.cwd().glob("Deeplearning/*/3장/ch3_utils.py")) + list(Path.cwd().glob("**/ch3_utils.py"))
    if matches:
        NOTEBOOK_DIR = matches[0].parent
    else:
        NOTEBOOK_DIR = Path("Deeplearning") / "Vision 응용" / "3장"
sys.path.insert(0, str(NOTEBOOK_DIR))

from ch3_utils import *

paths = find_ch3_paths()
set_korean_font()
set_seed(31)
paths

Chapter3Paths(chapter3_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장'), chapter2_2_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장'), data_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/data'), stress_ladder_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/data/synthetic_metal_stress_ladder'), runs_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/runs'), manifest_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/runs/manifests'), ch2_2_runs_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장/runs'))

## 33-1. feature bank 로드

In [2]:
FEATURE_VARIANT = "baseline_no_aug"
FEATURE_SEED = 0
feature_dir = paths.runs_root / "feature_bank" / FEATURE_VARIANT / f"seed_{FEATURE_SEED}"
feature_path = feature_dir / "stage_features.npz"
index_path = feature_dir / "sample_index.csv"
if not feature_path.exists() or not index_path.exists():
    raise FileNotFoundError("31번 노트북에서 feature bank를 먼저 생성하세요.")

index = pd.read_csv(index_path)
arrays = np.load(feature_path)
print("arrays:", arrays.files)
display(index.head())

arrays: ['stage1_global', 'stage1_target', 'stage1_background', 'stage2_global', 'stage2_target', 'stage2_background', 'stage3_global', 'stage3_target', 'stage3_background', 'stage4_global', 'stage4_target', 'stage4_background']


,sample_id,color_group,shape_group,defect_type,domain_block,stress_severity,target_dice,target_fnr,success
0,eval_matched_matched_control_neutral_scratch_t...,neutral,top_half_metal,scratch,matched_control,NaN,0.714286,0.257812,1
1,eval_matched_matched_control_neutral_scratch_t...,neutral,top_half_metal,scratch,matched_control,NaN,0.175953,0.818182,0
2,eval_matched_matched_control_neutral_scratch_t...,neutral,top_half_metal,scratch,matched_control,NaN,0.000000,1.000000,0
3,eval_matched_matched_control_neutral_scratch_t...,neutral,top_half_metal,scratch,matched_control,NaN,0.259819,0.839552,0
4,eval_matched_matched_control_neutral_scratch_t...,neutral,top_half_metal,scratch,matched_control,NaN,0.000000,1.000000,0


## 33-2. separability 계산과 시각화

In [3]:
out_dir = paths.runs_root / "feature_separability"
separability = summarize_feature_separability(feature_path, index_path, out_dir)
plot_feature_separability(separability, out_dir / "stage_feature_separability.png")
display(separability.sort_values("nearest_centroid_accuracy", ascending=False).head(20))

all_parts = []
for index_csv in sorted((paths.runs_root / "feature_bank").glob("*/seed_*/sample_index.csv")):
    variant = index_csv.parent.parent.name
    seed_name = index_csv.parent.name
    feature_npz = index_csv.parent / "stage_features.npz"
    if not feature_npz.exists():
        continue
    tmp_dir = out_dir / "_tmp" / variant / seed_name
    part = summarize_feature_separability(feature_npz, index_csv, tmp_dir)
    part["variant"] = variant
    part["seed"] = int(seed_name.split("_")[-1])
    all_parts.append(part)
all_sep = pd.concat(all_parts, ignore_index=True)
all_sep.to_csv(out_dir / "stage_feature_separability_all_models.csv", index=False, encoding="utf-8-sig")
display(
    all_sep[
        (all_sep["feature"].isin(["stage1_global", "stage2_global"]))
        & (all_sep["label"].isin(["color_group", "defect_type", "success"]))
    ]
    .groupby(["variant", "feature", "label"])["nearest_centroid_accuracy"]
    .mean()
    .reset_index()
    .sort_values(["feature", "label", "variant"])
)

,feature,label,n_samples,n_labels,nearest_centroid_accuracy,mean_pairwise_centroid_distance
47,stage4_global,shape_group,240,2,0.904762,0.413147
0,stage1_global,color_group,240,5,0.894118,1.544946
57,stage4_background,shape_group,240,2,0.869048,0.383331
10,stage1_background,color_group,240,5,0.858824,1.531121
9,stage1_target,success,240,2,0.797619,1.838013
34,stage3_global,success,240,2,0.797619,2.408194
44,stage3_background,success,240,2,0.785714,2.389644
19,stage2_global,success,240,2,0.773810,2.272661
24,stage2_target,success,240,2,0.773810,4.272009
15,stage2_global,color_group,240,5,0.764706,2.365951


,variant,feature,label,nearest_centroid_accuracy
0,baseline_no_aug,stage1_global,color_group,0.890196
6,photometric_aug,stage1_global,color_group,0.960784
1,baseline_no_aug,stage1_global,defect_type,0.186508
7,photometric_aug,stage1_global,defect_type,0.198413
2,baseline_no_aug,stage1_global,success,0.726190
8,photometric_aug,stage1_global,success,0.492063
3,baseline_no_aug,stage2_global,color_group,0.650980
9,photometric_aug,stage2_global,color_group,0.650980
4,baseline_no_aug,stage2_global,defect_type,0.309524
10,photometric_aug,stage2_global,defect_type,0.353175


## 33-3. 해석 포인트

In [4]:
color_rows = separability[separability["label"] == "color_group"].sort_values("feature")
defect_rows = separability[separability["label"] == "defect_type"].sort_values("feature")
display(color_rows)
display(defect_rows)

print("H-A 지지 패턴: stage1/2에서 color separability가 defect separability보다 높고, photometric_aug 모델에서 낮아져야 합니다.")
print("H-B 지지 패턴: scratch success/failure가 target pooled feature에서 분리되며, stage가 깊어질수록 target energy가 감소해야 합니다.")

,feature,label,n_samples,n_labels,nearest_centroid_accuracy,mean_pairwise_centroid_distance
10,stage1_background,color_group,240,5,0.858824,1.531121
0,stage1_global,color_group,240,5,0.894118,1.544946
5,stage1_target,color_group,240,5,0.647059,2.419922
25,stage2_background,color_group,240,5,0.752941,2.359265
15,stage2_global,color_group,240,5,0.764706,2.365951
20,stage2_target,color_group,240,5,0.552941,3.734887
40,stage3_background,color_group,240,5,0.494118,2.176502
30,stage3_global,color_group,240,5,0.505882,2.185108
35,stage3_target,color_group,240,5,0.317647,2.938526
55,stage4_background,color_group,240,5,0.423529,0.162619


,feature,label,n_samples,n_labels,nearest_centroid_accuracy,mean_pairwise_centroid_distance
11,stage1_background,defect_type,240,4,0.142857,0.077288
1,stage1_global,defect_type,240,4,0.202381,0.099841
6,stage1_target,defect_type,240,4,0.464286,1.516868
26,stage2_background,defect_type,240,4,0.214286,0.197074
16,stage2_global,defect_type,240,4,0.214286,0.219709
21,stage2_target,defect_type,240,4,0.690476,4.194739
41,stage3_background,defect_type,240,4,0.380952,0.767829
31,stage3_global,defect_type,240,4,0.380952,0.855816
36,stage3_target,defect_type,240,4,0.583333,4.998787
56,stage4_background,defect_type,240,4,0.333333,0.070697


H-A 지지 패턴: stage1/2에서 color separability가 defect separability보다 높고, photometric_aug 모델에서 낮아져야 합니다.
H-B 지지 패턴: scratch success/failure가 target pooled feature에서 분리되며, stage가 깊어질수록 target energy가 감소해야 합니다.
